In [21]:

import os
import pandas as pd
import numpy as np
import pywt
from scipy import signal, stats
from matplotlib import pyplot as plt
from matplotlib import font_manager, rcParams
import lightgbm as lgb
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import mean_absolute_error
import pathlib
from skimage.restoration import denoise_wavelet

# 设置中文字体（可选）
font_path = "../wqy-zenhei.ttc"
font_manager.fontManager.addfont(font_path)
font_prop = font_manager.FontProperties(fname=font_path)
rcParams['font.sans-serif'] = [font_prop.get_name()]
rcParams['axes.unicode_minus'] = False

In [22]:


# 采样参数
FS = 25600


WINDOW_SIZE = 5   # 滑动窗口大小（计算局部均值）
TREND_SIZE = 10   # 趋势计算窗口大小

# %%
def compute_psd(x, fs=FS, nperseg=None):
    if nperseg is None:
        nperseg = min(len(x), 4096)
    freqs, psd = signal.welch(x - np.mean(x), fs=fs, window='hann', nperseg=nperseg)
    return freqs, psd

def extract_time_features(x):
    rms = np.sqrt(np.mean(x**2))
    peak = np.max(np.abs(x))
    return {
        "rms": rms,
        "std": np.std(x),
        "mean": np.mean(x),
        "kurtosis": stats.kurtosis(x),
        "skewness": stats.skew(x),
        "peak2peak": np.ptp(x),
        "crest_factor": peak / (rms + 1e-12),
        "impulse_factor": peak / (np.mean(np.abs(x)) + 1e-12),
        "clearance_factor": peak / ((np.mean(np.sqrt(np.abs(x)))**2) + 1e-12),
    }

def extract_psd_features(x, fs=FS, n_bins=10):
    freqs, psd = compute_psd(x, fs)
    psd_energy = psd.sum() + 1e-12
    psd_norm = psd / psd_energy
    centroid = np.sum(freqs * psd_norm)
    bandwidth = np.sqrt(np.sum(((freqs - centroid)**2) * psd_norm))
    entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-12))
    flatness = np.exp(np.mean(np.log(psd + 1e-12))) / (np.mean(psd) + 1e-12)
    feats = {
        "psd_energy": psd_energy,
        "spectral_entropy": entropy,
        "spectral_centroid": centroid,
        "spectral_bandwidth": bandwidth,
        "spectral_flatness": flatness
    }
    # 分频能量
    fmax = freqs[-1]
    bins = np.linspace(0, fmax, n_bins+1)
    for i in range(n_bins):
        idx = (freqs >= bins[i]) & (freqs < bins[i+1])
        feats[f"psd_bin_energy_{i}"] = psd[idx].sum() / psd_energy
    return feats

def extract_envelope_features(x, fs=FS, n_bins=8):
    analytic = signal.hilbert(x - np.mean(x))
    env = np.abs(analytic)
    env -= env.mean()
    freqs, psd_env = compute_psd(env, fs)
    psd_env += 1e-12
    psd_env_norm = psd_env / psd_env.sum()
    centroid = np.sum(freqs * psd_env_norm)
    entropy = -np.sum(psd_env_norm * np.log(psd_env_norm + 1e-12))
    feats = {
        "env_rms": np.sqrt(np.mean(env**2)),
        "env_kurtosis": stats.kurtosis(env),
        "env_entropy": entropy,
        "env_centroid": centroid
    }
    fmax = freqs[-1]
    bins = np.linspace(0, fmax, n_bins+1)
    for i in range(n_bins):
        idx = (freqs >= bins[i]) & (freqs < bins[i+1])
        feats[f"env_bin_energy_{i}"] = psd_env[idx].sum() / psd_env.sum()
    return feats

def extract_wavelet_features(x):
    x = x - np.mean(x)
    coeffs = pywt.wavedec(x, 'db4', level=4)
    energies = np.array([np.sum(c**2) for c in coeffs])
    total_energy = energies.sum() + 1e-12
    high_ratio = (energies[-1] + energies[-2]) / total_energy
    energy_norm = energies / total_energy
    wavelet_entropy = -np.sum(energy_norm * np.log(energy_norm + 1e-12))
    return {
        "wavelet_energy": total_energy,
        "wavelet_high_ratio": high_ratio,
        "wavelet_entropy": wavelet_entropy
    }

def extract_features(x, fs=FS):
    # x = denoise_wavelet(x, method='BayesShrink', mode='soft', wavelet='db4', rescale_sigma=True)

    feats = {}
    feats.update(extract_time_features(x))
    feats.update(extract_psd_features(x, fs))
    feats.update(extract_envelope_features(x, fs))
    feats.update(extract_wavelet_features(x))
    return feats

In [23]:



def process_bearing_folder(folder, delta_t=10):
    acc_files = sorted([os.path.join(folder, f) for f in os.listdir(folder)
                        if f.startswith("acc") and f.endswith(".csv")])
    
    feats_list = []
    for file in acc_files:
        try:
            df = pd.read_csv(file, header=None, sep=',').to_numpy()
            sig = np.sqrt(df[:, 4]**2 + df[:, 5]**2)
        except:
            df = pd.read_csv(file, header=None, sep=';').to_numpy()
            sig = np.sqrt(df[:, 4]**2 + df[:, 5]**2)
        # 对水平和垂直进行均方根
        
        # try:
        #     sig = pd.read_csv(file, header=None, sep=',').to_numpy()[:,4]
        # except:
        #     sig = pd.read_csv(file, header=None, sep=';').to_numpy()[:,4]
        # TODO：这里可以考虑增加一些数据清洗步骤，比如去除异常值、平滑信号等
        feats_list.append(extract_features(sig))
    df_feats = pd.DataFrame(feats_list)
    
    # 1. 
    # df_smooth = df_feats.ewm(alpha=0.3).mean()
    # df_smooth.columns = [f"{c}_smooth" for c in df_smooth.columns]
    # df_feats = pd.concat([df_feats, df_smooth], axis=1)
   
    # 2. RUL 标签（真实秒数，不泄漏未来信息）
    rul = np.arange(len(acc_files)-1, -1, -1) * delta_t
    df_feats['RUL'] = rul
    
   
    # 3. 
    sub = pathlib.Path(folder).stem
    df_feats['bearing_id'] = sub
    # 4. 增加bearing type反而效果会下降
    # if sub.startswith("Bearing1_"):
    #     df_feats['BearType'] = 1
    # elif sub.startswith("Bearing2_"):
    #     df_feats['BearType'] = 2
    # elif sub.startswith("Bearing3_"):
    #     df_feats['BearType'] = 3
    # else:
    #     raise ValueError(f"Unexpected bearing name: {sub}")
    
    return df_feats

def build_train_dataset(data_path):
    all_dfs = []
    print("Processing training data...")
    for sub in os.listdir(data_path):
        folder = os.path.join(data_path, sub)
        if not os.path.isdir(folder): continue
        print(f"  -> {sub}")
        df = process_bearing_folder(folder)
        all_dfs.append(df)

    full_df = pd.concat(all_dfs).reset_index(drop=True)
    
    feature_cols = [c for c in full_df.columns if c not in ['RUL', 'bearing_id']]
    
    X_list = []
    y_list = []
    groups_list = []
    
    for bid, group in full_df.groupby('bearing_id'):
        group = group.reset_index(drop=True)
        # group = group.iloc[50:]
        n = len(group)
        cut_ratio = 0.3
        cut_idx = int(n * cut_ratio)
        group = group.iloc[cut_idx:].reset_index(drop=True)
        # 基础特征
        curr_feats = group[feature_cols]
        
        # 1. 差分特征
        diff_feats = curr_feats.diff().fillna(0)
        diff_feats.columns = [f"{c}_diff" for c in feature_cols]
        # 2. 
        roll_mean = curr_feats.rolling(window=WINDOW_SIZE, min_periods=1).mean()
        roll_mean.columns = [f"{c}_roll_mean" for c in feature_cols]
        
        roll_max = curr_feats.rolling(window=WINDOW_SIZE, min_periods=1).max()
        roll_max.columns = [f"{c}_roll_max" for c in feature_cols]
        
        roll_min = curr_feats.rolling(window=WINDOW_SIZE, min_periods=1).min()
        roll_min.columns = [f"{c}_roll_min" for c in feature_cols]
        
        # 3. 局部趋势 (简单用当前值减去N个点前的值代替斜率，计算更快)
        trend_feats = curr_feats - curr_feats.shift(TREND_SIZE).bfill()
        trend_feats.columns = [f"{c}_trend" for c in feature_cols]
    
        # 合并特征
        X_group = pd.concat([curr_feats, diff_feats, roll_mean,roll_max, roll_min, trend_feats], axis=1)

        y_group = group['RUL']
        y_group_log = np.log1p(y_group)
        X_list.append(X_group)
        y_list.append(y_group_log)
        groups_list.extend([bid] * len(X_group))

    X = pd.concat(X_list, axis=0).reset_index(drop=True)
    y = pd.concat(y_list, axis=0).reset_index(drop=True)
    groups = np.array(groups_list)
    return X, y, groups

base_path = "./phm-ieee-2012-data-challenge-dataset-master/Learning_set"
X, y, groups = build_train_dataset(base_path)
# LightGBM 参数

params = {
    'learning_rate': 0.02,
          'boosting_type': 'gbdt',
          'objective': 'regression', 
          'metric': 'mae',
          'num_leaves': 67, 
          'verbose': -1,
          'seed': 2222, 
          'n_jobs': 32,
          'min_child_weight': 9, 
          'max_depth': 6,
          'lambda_l1': 0.8,
          'lambda_l2': 1.5,
          'feature_fraction': 0.5, 
          'bagging_fraction': 0.95, 
          'bagging_freq': 5, 
          }


gkf = GroupKFold(n_splits=5)
models = []
oof_preds = np.zeros(len(X))
scores = []

for fold, (trn_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_train, y_train = X.iloc[trn_idx], y.iloc[trn_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    print(f"Fold {fold+1} Train on: {np.unique(groups[trn_idx])} Validating on: {np.unique(groups[val_idx])}")
    lgb_train = lgb.Dataset(X_train, y_train)
    lgb_val = lgb.Dataset(X_val, y_val)
    
    model = lgb.train(params, train_set=lgb_train,
                      valid_sets=[lgb_train, lgb_val],
                      valid_names=['train', 'valid'],
                      num_boost_round=3000,
                      callbacks=[lgb.early_stopping(50, first_metric_only=True), lgb.log_evaluation(0)])
    
    val_pred = np.expm1(model.predict(X_val))
    y_val_true = np.expm1(y_val)
    
    score = mean_absolute_error(y_val_true, val_pred)
    
    oof_preds[val_idx] = val_pred
    
    scores.append(score)
    models.append(model)
    print(f"Fold {fold+1} MAE: {score:.2f}")
print(f"Average MAE: {np.mean(scores):.2f}")


Processing training data...
  -> Bearing1_2
  -> Bearing3_2
  -> Bearing1_1
  -> Bearing2_1
  -> Bearing2_2
  -> Bearing3_1
Fold 1 Train on: ['Bearing1_2' 'Bearing2_1' 'Bearing2_2' 'Bearing3_1' 'Bearing3_2'] Validating on: ['Bearing1_1']
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[46]	train's l1: 0.442928	valid's l1: 1.13931
Evaluated only: l1
Fold 1 MAE: 6967.11
Fold 2 Train on: ['Bearing1_1' 'Bearing1_2' 'Bearing2_1' 'Bearing2_2' 'Bearing3_1'] Validating on: ['Bearing3_2']
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[55]	train's l1: 0.402211	valid's l1: 0.62675
Evaluated only: l1
Fold 2 MAE: 2707.74
Fold 3 Train on: ['Bearing1_1' 'Bearing1_2' 'Bearing2_2' 'Bearing3_1' 'Bearing3_2'] Validating on: ['Bearing2_1']
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.869338	valid's l1: 0.733829
Evaluated only: l1
Fold 3 MAE:

In [24]:
# import shap, tqdm

# def view_top_features():
#     for index, model in tqdm.tqdm(enumerate(models)):
#         explainer = shap.TreeExplainer(model)
#         shap_values = explainer.shap_values(X)
#         feature_importance = np.abs(shap_values).mean(axis=0)
#         feat_imp_df = pd.DataFrame({
#             'feature': X.columns,
#             'importance': feature_importance
#         }).sort_values(by='importance', ascending=False)
        
#         top100 = feat_imp_df.head(100)
#         plt.figure(figsize=(18, 30)) 
#         plt.barh(top100['feature'], top100['importance'])
        
#         plt.title(f'Fold {index} Feature Importance')
#         plt.xlabel("Importance")
#         plt.ylabel("Feature")
        
#         plt.gca().invert_yaxis()  # 让重要性高的在上方
#         plt.tight_layout()
#         plt.show()
# view_top_features()

In [25]:
import shap


from collections import defaultdict

feature_importance_score = defaultdict(list)
for model in models:
    feature_names = model.feature_name()
    # importances = model.feature_importance(importance_type='gain')
    # for name, imp in zip(feature_names, importances):
    #     feature_importance_score[name].append(imp)
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    importance = np.abs(shap_values).mean(axis=0)
    for name, imp in zip(feature_names, importance):
        feature_importance_score[name].append(imp)

feature_agg = {feat: np.median(imps) for feat, imps in feature_importance_score.items()}
feat_imp_df = pd.DataFrame({
    'feature': list(feature_agg.keys()),
    'importance': list(feature_agg.values())
}).sort_values('importance', ascending=False)

top_n = 100
top_features = feat_imp_df.head(top_n)['feature'].tolist()

corr_matrix = X[top_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] >= 0.9)]
top_features = [f for f in top_features if f not in to_drop]
print(len(top_features))
print(top_features)


24
['wavelet_entropy_roll_mean', 'wavelet_entropy_roll_min', 'wavelet_entropy_roll_max', 'std_roll_min', 'env_bin_energy_6_roll_mean', 'psd_bin_energy_1_roll_max', 'wavelet_high_ratio_roll_max', 'skewness_roll_min', 'psd_bin_energy_2_roll_max', 'peak2peak_roll_max', 'psd_bin_energy_9_roll_mean', 'psd_bin_energy_3_roll_mean', 'mean_trend', 'psd_bin_energy_4_roll_min', 'skewness_roll_mean', 'env_bin_energy_1_roll_mean', 'kurtosis_roll_min', 'kurtosis_roll_max', 'env_bin_energy_7_roll_mean', 'env_bin_energy_1_roll_min', 'env_bin_energy_7_roll_min', 'env_bin_energy_2_roll_min', 'wavelet_high_ratio_roll_min', 'env_bin_energy_6_roll_min']


In [26]:
X_selected = X[top_features]

final_models = []
scores = []

for fold, (trn_idx, val_idx) in enumerate(gkf.split(X_selected, y, groups)):

    X_train, y_train = X_selected.iloc[trn_idx], y.iloc[trn_idx]
    X_val, y_val = X_selected.iloc[val_idx], y.iloc[val_idx]

    train_set = lgb.Dataset(X_train, y_train)
    val_set = lgb.Dataset(X_val, y_val)

    model = lgb.train(params, train_set=train_set,
                      valid_sets=[train_set, val_set],
                      valid_names=['train', 'valid'],
                      num_boost_round=3000,
                      callbacks=[lgb.early_stopping(50, first_metric_only=True), lgb.log_evaluation(0)])

    val_pred = np.expm1(model.predict(X_val))
    y_val_true = np.expm1(y_val)

    score = mean_absolute_error(y_val_true, val_pred)

    scores.append(score)
    final_models.append(model)

    print(f"[Refit] Fold {fold+1} MAE: {score:.2f}")

print("Final CV MAE:", np.mean(scores))

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[52]	train's l1: 0.435133	valid's l1: 1.24563
Evaluated only: l1
[Refit] Fold 1 MAE: 7365.76
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[82]	train's l1: 0.326486	valid's l1: 0.684846
Evaluated only: l1
[Refit] Fold 2 MAE: 3034.96
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	train's l1: 0.842362	valid's l1: 0.735301
Evaluated only: l1
[Refit] Fold 3 MAE: 1735.63
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[19]	train's l1: 0.661923	valid's l1: 0.653148
Evaluated only: l1
[Refit] Fold 4 MAE: 1442.57
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[81]	train's l1: 0.297747	valid's l1: 0.762831
Evaluated only: l1
[Refit] Fold 5 MAE: 1519.54
Final CV MAE: 3019.691445575678


In [27]:

def predict_test_folder(folder, models, top_features):
    df = process_bearing_folder(folder)
    feature_cols = [c for c in df.columns if c not in ['RUL', 'bearing_id']]

    
    curr_feats = df[feature_cols].copy()
    diff_feats = curr_feats.diff().fillna(0)
    diff_feats.columns = [f"{c}_diff" for c in feature_cols]

    roll_mean = curr_feats.rolling(window=WINDOW_SIZE, min_periods=1).mean()
    roll_mean.columns = [f"{c}_roll_mean" for c in feature_cols]
    
    roll_max = curr_feats.rolling(window=WINDOW_SIZE, min_periods=1).max()
    roll_max.columns = [f"{c}_roll_max" for c in feature_cols]
    
    roll_min = curr_feats.rolling(window=WINDOW_SIZE, min_periods=1).min()
    roll_min.columns = [f"{c}_roll_min" for c in feature_cols]
    
    trend_feats = curr_feats - curr_feats.shift(TREND_SIZE).bfill()
    trend_feats.columns = [f"{c}_trend" for c in feature_cols]

    X_test_full = pd.concat([curr_feats, diff_feats, roll_mean, roll_max,roll_min,trend_feats], axis=1)
    # 我们只关心最后一个时刻的预测值（即当前剩余寿命）
    # 为了稳健，我们取最后 3 个点的预测均值
    X_last = X_test_full.iloc[-3:][top_features] 

    preds = []
    for model in models:
        p = np.expm1(model.predict(X_last))
        preds.append(np.mean(p))
    return np.mean(preds)
    

# 测试集预测
test_base = "./phm-ieee-2012-data-challenge-dataset-master/Full_Test_Set"
# 真实标签
test_rul_map = {
    "Bearing1_3": 5730,
    "Bearing1_4": 339,
    "Bearing1_5": 1610,
    "Bearing1_6": 1460,
    "Bearing1_7": 7570,
    "Bearing2_3": 7530,
    "Bearing2_4": 1390,
    "Bearing2_5": 3090,
    "Bearing2_6": 1290,
    "Bearing2_7": 580,
    "Bearing3_3": 820,
}

test_rul_pred_map = {}
for sub in os.listdir(test_base):
    folder = os.path.join(test_base, sub)
    if not os.path.isdir(folder): continue
    pred_rul = predict_test_folder(folder, final_models, top_features=top_features)
    test_rul_pred_map[sub] = pred_rul
    print(f"{sub}: Predicted RUL = {pred_rul:.2f} seconds, True RUL = {test_rul_map[sub]} seconds")

total_error = 0
for bearing, y_true in test_rul_map.items():
    y_pred = test_rul_pred_map[bearing]
    total_error += abs(float(y_true) - y_pred)
    
print(f'{total_error=}')


Bearing3_3: Predicted RUL = 1035.76 seconds, True RUL = 820 seconds
Bearing1_3: Predicted RUL = 1057.58 seconds, True RUL = 5730 seconds
Bearing2_6: Predicted RUL = 1086.73 seconds, True RUL = 1290 seconds
Bearing1_7: Predicted RUL = 939.65 seconds, True RUL = 7570 seconds
Bearing2_4: Predicted RUL = 1462.57 seconds, True RUL = 1390 seconds
Bearing1_4: Predicted RUL = 1027.08 seconds, True RUL = 339 seconds
Bearing2_7: Predicted RUL = 2024.14 seconds, True RUL = 580 seconds
Bearing2_3: Predicted RUL = 1485.21 seconds, True RUL = 7530 seconds
Bearing2_5: Predicted RUL = 2053.91 seconds, True RUL = 3090 seconds
Bearing1_5: Predicted RUL = 1139.87 seconds, True RUL = 1610 seconds
Bearing1_6: Predicted RUL = 1124.98 seconds, True RUL = 1460 seconds
total_error=np.float64(21812.644849579287)
